# 10.17 — Autoregressive generation (PixelCNN, WaveNet)
Autoregressive models generate one variable at a time, multiplying simple conditional probabilities into a full joint distribution. PixelCNN and WaveNet enforce this with masked or causal context, trading exact likelihood for slower sampling. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
The chain rule factorizes $p(x_1,\ldots,x_D)=\prod_i p(x_i\mid x_{<i})$, so $\log p(x)=\sum_i\log p(x_i\mid x_{<i})$. For $(0.5,0.7,0.2)$, the lesson probability is $0.0700$ and NLL is $2.659$.

In [ ]:

def autoregressive_logprob_and_sample(conditional_probs):
    probs = np.asarray(conditional_probs, dtype=float)
    sequence_probability = float(np.prod(probs))
    nll = float(-np.sum(np.log(probs)))
    sample = (probs >= 0.5).astype(int)
    return sequence_probability, nll, sample

sequence_probability, nll, sample = autoregressive_logprob_and_sample([0.5, 0.7, 0.2])
assert round(sequence_probability, 4) == 0.0700
assert round(nll, 3) == 2.659
print("probability", round(sequence_probability, 4), "nll", round(nll, 3), "sample", sample.tolist())


The ladder model below discretizes each dimension and learns conditional histograms in raster order. It is small, CPU-only, and still respects the real autoregressive rule: future dimensions are unavailable at prediction time.

In [ ]:

def discretize01(data, bins=8):
    x = normalize01(data)
    q = np.clip((x * bins).astype(int), 0, bins - 1)
    return q


def train_ar_histogram(data, bins=8, alpha=0.5):
    q = discretize01(data, bins=bins)
    n, dim = q.shape
    tables = []
    for j in range(dim):
        if j == 0:
            counts = np.bincount(q[:, j], minlength=bins).astype(float) + alpha
            tables.append({"type": "marginal", "probs": counts / counts.sum()})
        else:
            prev = q[:, j - 1]
            table = np.full((bins, bins), alpha, dtype=float)
            for row in range(n):
                table[prev[row], q[row, j]] += 1.0
            table = table / table.sum(axis=1, keepdims=True)
            tables.append({"type": "conditional", "probs": table})
    return tables


def ar_nll_and_sample(data, bins=8, seed=SEED):
    local_rng = np.random.default_rng(seed)
    q = discretize01(data, bins=bins)
    tables = train_ar_histogram(data, bins=bins)
    total = 0.0
    count = 0
    for row in q:
        for j, table in enumerate(tables):
            if j == 0:
                prob = table["probs"][row[j]]
            else:
                prob = table["probs"][row[j - 1], row[j]]
            total -= math.log(float(prob) + 1e-12)
            count += 1
    sample = np.zeros(data.shape[1], dtype=int)
    for j, table in enumerate(tables):
        if j == 0:
            probs = table["probs"]
        else:
            probs = table["probs"][sample[j - 1]]
        sample[j] = local_rng.choice(np.arange(bins), p=probs)
    sample = sample.astype(float) / max(1, bins - 1)
    return total / count, sample


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    metric, sample = ar_nll_and_sample(rung["data"], bins=8, seed=SEED + level)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": sample, "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: NLL per variable={metric:.5f}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("NLL per variable")
plt.xlabel("F9 rung")
plt.title("Autoregressive histogram: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: leaking future pixels or samples
If $x_i$ sees $x_{>i}$, the likelihood is invalid. We reproduce a leaked D5 score by conditioning each variable on itself, then fix it with causal raster-order conditioning.

In [ ]:

def leaked_future_nll(data, bins=8):
    q = discretize01(data, bins=bins)
    counts = np.zeros((data.shape[1], bins), dtype=float) + 0.5
    for j in range(data.shape[1]):
        counts[j] += np.bincount(q[:, j], minlength=bins)
    probs = counts / counts.sum(axis=1, keepdims=True)
    leaked = 0.0
    count = 0
    for row in q:
        for j in range(data.shape[1]):
            prob = max(0.98, probs[j, row[j]])
            leaked -= math.log(float(prob))
            count += 1
    return leaked / count

d5 = ladder[-1]
leaked = leaked_future_nll(d5["data"])
causal, causal_sample = ar_nll_and_sample(d5["data"], bins=8)
print(f"leaked invalid NLL={leaked:.5f}")
print(f"causal valid NLL={causal:.5f}")
fig, axes = plt.subplots(1, 2, figsize=(5, 2.4))
preview_array(axes[0], d5["data"][0], d5["shape"], "target D5")
preview_array(axes[1], causal_sample, d5["shape"], "causal sample")
plt.tight_layout()


## Evaluate it + Practice
- Track `NLL` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: let the model condition on the current or future value, which makes likelihood look too good; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: reverse raster order and compare the NLL on D4 digits.

Practice 3: increase the number of bins and explain the likelihood-data sparsity tradeoff.